# 04 Spot checking: 8 algorithms, default hyper-parameters, spatial 5-fold CV

Ridge, KNN, decision tree, random forest, SVR, HistGB, MLP (and the weighted variants if you set `WEIGHTING`) on the
shipped features table, with the same spatial folds as the rest of the paper:
`StratifiedGroupKFold(5, shuffle, seed 42)`, groups = city + 2,560 m block, strata = height classes [2, 6, 15, 30] m.
Imputation, scaling and the feature reduction are fitted inside each training fold only; `test` and `excluded` rows are
never used for fitting or validation.

**Input:** `data/features_full_1872_split.(parquet|csv)`  (`python scripts/download_data.py`).
**Output:** `outputs/spot_check_8cities/` (OOF predictions, summary metrics, charts).
**Time:** hours on a laptop with `FAST = False` (SVR about 10 min per fold, random forest about 5 min per fold on an Apple M-series).
Set `FAST = True` for a small subsample that checks the plumbing in minutes.

In [ ]:
# Setup: make `config.py` importable whether Jupyter was started in the repo root or in notebooks/
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, _here / "notebooks", _here.parent / "notebooks"):
    if (_p / "config.py").exists():
        sys.path.insert(0, str(_p)); break
else:
    raise FileNotFoundError("config.py not found: start Jupyter from the repository root or from notebooks/")
import config
print("repo root:", config.ROOT)

In [ ]:
# =====================================================================
# SPOT-CHECK CELL (CMP263): 8 candidate pipelines, same spatial folds,
# repeated runs, fixed seeds, RMSE charts incl. Frantz-style
# frequency-weighted RMSE.  Paste into ONE notebook cell and run.
#
# Spot-checking = quick look at many algorithms with DEFAULT hyper-
# parameters (no tuning).  Tuning, Frantz step 2 and stacking come later.
#
# Leakage rules followed here: imputer, scaler and the feature reduction
# are fit on the TRAINING rows of each fold only, then applied (transform
# only) to that fold's validation rows.  Folds are spatial blocks, so a
# block is never split between train and validation.  Rows with
# split == "test" / "excluded" are never used for fitting or validation;
# the natural test rows are read ONLY for their height histogram, which
# gives the population frequencies behind the weighted RMSE (see below).
#
# =====================================================================
import os, time, warnings, inspect
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")

# ------------------------- CONFIG ------------------------------------
DATA = config.OUT
TABLE = str(config.TABLE)
EXTRA_TRAIN_SPLITS = []   # [] = the reported run. ["unused"] adds the rows the rebalancing left out to every training fold (rejected experiment, see README)
                          # Validation rows and folds stay exactly those of the plain run; extra rows in a validation block are dropped (no spatial leakage).
WEIGHTING = None       # None | "inv_slice" | "sqrt_inv_slice": sample weights from how rare a row's 1 m height slice is (see slice_weights)
W_MAX = 4.0            # cap on a row's weight before normalising to mean 1
TOP_EQUIV_SLICES = 5   # the pooled top slice (>= 49 m) spans many metres: count it as this many 1 m slices when measuring rarity
# every WEIGHTING gets its own folder, so the runs can be compared; the unweighted run keeps the plain name
OUT = (f"{DATA}/spot_check_8cities" + ("_plus_" + "_".join(EXTRA_TRAIN_SPLITS) if EXTRA_TRAIN_SPLITS else "")
       + (f"_w_{WEIGHTING}_cap{W_MAX:g}" if WEIGHTING else ""))
CITIES = None          # e.g. ["porto_alegre", "sao_paulo"]; None = all cities in the table
N_FOLDS = 5
N_REPEATS = 1          # first run on the 8-city table: 1 (about 17k rows); raise to 2 once the timings are known. Each repeat reshuffles which blocks go to which fold (same for all models)
BLOCK_M = 2560.0       # spatial block size, metres
SEED = 42
FAST = False           # True = small subsample + few trees, to test the plumbing in minutes
N_JOBS = -1
SKIP = ["SVR_full"]    # slowest model (all 1,872 features, ~13k rows per fold); remove it from SKIP for a later run, finished parts are reused
# ---------------------------------------------------------------------

META = ["sample_id", "city", "height_m", "is_background", "x_m", "y_m", "split"]
CLASS_EDGES = [2, 6, 15, 30]                    # 5 classes: <2, 2-6, 6-15, 15-30, >30 m
CLASS_NAMES = ["<2 m", "2-6 m", "6-15 m", "15-30 m", ">30 m"]
FINE_EDGES = [2, 4, 6, 9, 12, 15, 20, 30, 50]   # 10 finer bins for the charts
FINE_NAMES = ["<2", "2-4", "4-6", "6-9", "9-12", "12-15", "15-20", "20-30", "30-50", ">50"]
MIN_BIN_N = 10                                   # bins with fewer pooled samples are not drawn

# Categorical palette (validated reference order, light surface); slot per model, fixed
SLOT = {"SVR": "#2a78d6", "RandomForest": "#eb6834", "HistGB": "#1baf7a", "SVR_full": "#eda100",
        "KNN": "#e87ba4", "Ridge": "#008300", "MLP": "#6250d6", "DecisionTree": "#e34948"}
MARK = {"SVR": "o", "RandomForest": "s", "HistGB": "^", "SVR_full": "D",
        "KNN": "v", "Ridge": "P", "MLP": "X", "DecisionTree": "*"}
LINE = {"SVR": "-", "RandomForest": "-", "HistGB": "-", "SVR_full": "--",
        "KNN": "--", "Ridge": "--", "MLP": ":", "DecisionTree": ":"}
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"


# ------------------- Frantz step 1 as a fit/transform step ------------
class RFThreshold(BaseEstimator, TransformerMixin):
    """Frantz et al. (2021) supplement A2, step 1: random forest regression on a random
    10% subsample (500 trees, 1% of features per split); keep features whose importance
    is above 1 per mille.  Fit on training rows only.  If fewer than `min_keep` survive
    the top `min_keep` are kept (safety net, reported)."""
    def __init__(self, frac=0.10, n_trees=500, mtry=0.01, thr=1e-3, min_keep=20,
                 min_rows=500, seed=0, n_jobs=-1):
        self.frac, self.n_trees, self.mtry, self.thr = frac, n_trees, mtry, thr
        self.min_keep, self.min_rows, self.seed, self.n_jobs = min_keep, min_rows, seed, n_jobs

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n = len(y)
        idx = rng.choice(n, size=min(n, max(int(self.frac * n), self.min_rows)), replace=False)
        rf = RandomForestRegressor(n_estimators=self.n_trees, max_features=self.mtry,
                                   n_jobs=self.n_jobs, random_state=self.seed).fit(X[idx], y[idx])
        imp = rf.feature_importances_
        keep = np.where(imp > self.thr)[0]
        self.fallback_ = len(keep) < self.min_keep
        if self.fallback_:
            keep = np.argsort(imp)[::-1][:self.min_keep]
        self.keep_ = np.sort(keep)
        return self

    def transform(self, X):
        return X[:, self.keep_]


# ------------------- candidate models (default hyper-parameters) ------
def model_zoo():
    f = FAST
    ttr = lambda est: TransformedTargetRegressor(regressor=est, transformer=StandardScaler())
    # name -> (factory, input matrix)   red = after Frantz step 1, full = all features
    return {
        "Ridge":        (lambda: Ridge(alpha=10.0), "red"),
        "KNN":          (lambda: KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=N_JOBS), "red"),
        "SVR":          (lambda: ttr(SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale")), "red"),
        "MLP":          (lambda: ttr(MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True,
                                                  max_iter=100 if f else 300, random_state=SEED)), "red"),
        "SVR_full":     (lambda: ttr(SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale")), "full"),
        "DecisionTree": (lambda: DecisionTreeRegressor(min_samples_leaf=5, random_state=SEED), "full"),
        "RandomForest": (lambda: RandomForestRegressor(n_estimators=60 if f else 300, max_features=0.33,
                                                       n_jobs=N_JOBS, random_state=SEED), "full"),
        "HistGB":       (lambda: HistGradientBoostingRegressor(max_iter=50 if f else 200, learning_rate=0.1,
                                                               early_stopping=False, random_state=SEED), "full"),
    }


# ------------------- sample weights ------------------------------------
def slice_weights(y):
    """Weights for the TRAINING rows of one fold (never uses validation rows).
    Rarity of a row's 1 m height slice (3-4 m ... 48-49 m; the pooled top slice counts as TOP_EQUIV_SLICES slices;
    rows below 3 m, i.e. background, are their own group): w = median slice count in 3-30 m / slice count.
    'sqrt_inv_slice' takes the square root. Weights are capped to [1, W_MAX] and normalised to mean 1, so the total
    weight (and with it the effective C of an SVR) stays comparable to the unweighted fit."""
    if not WEIGHTING:
        return None
    y = np.asarray(y, float)
    s = np.where(y < 3, -1, np.minimum(np.floor(y).astype(int), 49))
    cnt = pd.Series(s).value_counts()
    ref = float(np.median([cnt.get(k, 0) for k in range(3, 30) if cnt.get(k, 0) > 0]))
    eff = cnt.astype(float)
    if 49 in eff.index:
        eff[49] = eff[49] / TOP_EQUIV_SLICES
    w_slice = ref / eff
    if WEIGHTING == "sqrt_inv_slice":
        w_slice = np.sqrt(w_slice)
    elif WEIGHTING != "inv_slice":
        raise SystemExit(f"Unknown WEIGHTING {WEIGHTING!r}")
    w = np.clip(w_slice.reindex(s).to_numpy(), 1.0, W_MAX)
    w[s == -1] = 1.0                                   # background rows: unweighted
    return w / w.mean()


def accepts_weights(est):
    inner = getattr(est, "regressor", est)             # TransformedTargetRegressor wraps the model
    return "sample_weight" in inspect.signature(inner.fit).parameters


# ------------------- data ---------------------------------------------
def load_spot_table():
    df = config.read_table()
    if "split" not in df.columns:
        raise SystemExit("The table has no 'split' column: use the merged file with train/test labels.")
    if CITIES:
        df = df[df.city.isin(CITIES)]
    feats = [c for c in df.columns if c not in META and c not in
             {"system:index", ".geo", "n_parts_50m", "slice_lo"}]
    df[feats] = df[feats].apply(pd.to_numeric, errors="coerce")
    tr = df[df.split == "train"].dropna(subset=["height_m"]).reset_index(drop=True)
    te = df[df.split == "test"].dropna(subset=["height_m"]).reset_index(drop=True)
    print(f"train rows {len(tr)} | test rows (labels used only as weights) {len(te)} | "
          f"excluded {int((df.split == 'excluded').sum())} dropped | {len(feats)} features")
    print(tr.groupby("city").size().to_string())
    ex = df[df.split.isin(EXTRA_TRAIN_SPLITS)].dropna(subset=["height_m"]).reset_index(drop=True)
    if EXTRA_TRAIN_SPLITS:
        print(f"extra training rows {EXTRA_TRAIN_SPLITS}: {len(ex)} (added to every fold's training set, never validated)")
        print(ex.groupby("city").size().to_string())
    return tr, te, feats, ex


def population_fractions(te, cities):
    """Share of natural-test points per fine height bin, per city (the population frequencies)."""
    out = {}
    for c in cities:
        h = te.loc[te.city == c, "height_m"].to_numpy(float)
        if len(h) == 0:
            continue
        out[c] = np.bincount(np.digitize(h, FINE_EDGES), minlength=len(FINE_EDGES) + 1) / len(h)
    return out


# ------------------- metrics -------------------------------------------
def _wmse_city(y, p, fine, f):
    mse, wt = [], []
    for b in range(len(FINE_EDGES) + 1):
        m = fine == b
        if m.any() and f[b] > 0:
            mse.append(np.mean((p[m] - y[m]) ** 2)); wt.append(f[b])
    return np.average(mse, weights=wt) if mse else np.nan


def metric_row(y, p, city, fr):
    e = p - y
    cls = np.digitize(y, CLASS_EDGES)
    fine = np.digitize(y, FINE_EDGES)
    per_cls = [np.sqrt(np.mean(e[cls == c] ** 2)) for c in np.unique(cls)]
    wm = [_wmse_city(y[city == c], p[city == c], fine[city == c], fr[c]) for c in np.unique(city) if c in fr]
    return dict(RMSE=float(np.sqrt(np.mean(e ** 2))), MAE=float(np.mean(np.abs(e))),
                RMSE_macro=float(np.mean(per_cls)),
                RMSE_weighted=float(np.sqrt(np.nanmean(wm))) if wm else np.nan)


def per_bin_table(d):
    fine = np.digitize(d.y.to_numpy(), FINE_EDGES)
    rows = []
    for b in range(len(FINE_EDGES) + 1):
        m = fine == b
        if not m.any():
            continue
        y, p = d.y.to_numpy()[m], d.p.to_numpy()[m]
        rows.append(dict(bin=FINE_NAMES[b], bin_i=b, n=int(m.sum()), rmse=float(np.sqrt(np.mean((p - y) ** 2))),
                         bias=float(np.mean(p - y)), mean_ref=float(y.mean()), mean_pred=float(p.mean()),
                         mape=float(np.mean(np.abs(p - y) / y)) if y.min() >= 2 else np.nan))
    return pd.DataFrame(rows)


# ------------------- the spot-check loop -------------------------------
def run_cv(tr, feats, ex):
    os.makedirs(f"{OUT}/parts", exist_ok=True)
    # Resume guard: cached parts are reused only if the settings that change the results match.
    # (N_REPEATS and SKIP may change freely: more repeats just add parts.)
    import json, shutil
    fp = dict(EXTRA_TRAIN_SPLITS=EXTRA_TRAIN_SPLITS, n_extra=len(ex), WEIGHTING=WEIGHTING, W_MAX=W_MAX, TOP_EQUIV_SLICES=TOP_EQUIV_SLICES, FAST=FAST, N_FOLDS=N_FOLDS, BLOCK_M=BLOCK_M, SEED=SEED, CITIES=CITIES, TABLE=TABLE,
              table_bytes=os.path.getsize(TABLE), n_train=len(tr), n_features=len(feats))
    fp_path = f"{OUT}/parts/config.json"
    if os.path.exists(fp_path) and json.load(open(fp_path)) != fp:
        old = f"{OUT}/parts_old_{time.strftime('%Y%m%d_%H%M%S')}"
        shutil.move(f"{OUT}/parts", old); os.makedirs(f"{OUT}/parts")
        print(f"Settings changed since the cached run: old parts moved to {old}, starting clean.")
    elif not os.path.exists(fp_path) and any(f.endswith(".csv") for f in os.listdir(f"{OUT}/parts")):
        old = f"{OUT}/parts_old_unlabeled_{time.strftime('%Y%m%d_%H%M%S')}"
        shutil.move(f"{OUT}/parts", old); os.makedirs(f"{OUT}/parts")
        print(f"Cached parts without a settings record: moved to {old}, starting clean.")
    json.dump(fp, open(fp_path, "w"), indent=1)
    X = tr[feats].to_numpy(np.float32)
    y = tr["height_m"].to_numpy(float)
    names = np.array(feats)
    gx, gy = np.floor(tr.x_m / BLOCK_M).astype(int), np.floor(tr.y_m / BLOCK_M).astype(int)
    groups = (tr.city.astype(str) + "_" + gx.astype(str) + "_" + gy.astype(str)).to_numpy()
    strata = np.digitize(y, CLASS_EDGES)
    Xex = ex[feats].to_numpy(np.float32)
    yex = ex["height_m"].to_numpy(float)
    gex = (ex.city.astype(str) + "_" + np.floor(ex.x_m / BLOCK_M).astype(int).astype(str) + "_"
           + np.floor(ex.y_m / BLOCK_M).astype(int).astype(str)).to_numpy() if len(ex) else np.array([], dtype=object)
    zoo = {m: v for m, v in model_zoo().items() if m not in SKIP}
    log = []
    for rep in range(N_REPEATS):
        cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + rep)
        for k, (itr, ite) in enumerate(cv.split(X, strata, groups)):
            if FAST:
                itr = np.random.default_rng(SEED + rep).choice(itr, size=min(len(itr), 1500), replace=False)
            if len(ex):                      # extra training rows, minus those in a validation block
                keep = ~np.isin(gex, np.unique(groups[ite]))
                Xa, ya = np.vstack([X[itr], Xex[keep]]), np.concatenate([y[itr], yex[keep]])
            else:
                Xa, ya = X[itr], y[itr]
            todo = [m for m in zoo if not os.path.exists(f"{OUT}/parts/r{rep}_f{k}_{m}.csv")]
            if not todo:
                print(f"rep {rep} fold {k}: already done, skipped"); continue
            t0 = time.time()
            # --- everything below is fit on the training rows only ---
            imp = SimpleImputer(strategy="median").fit(Xa)
            sc = StandardScaler().fit(imp.transform(Xa))
            Xtr_full = sc.transform(imp.transform(Xa)); Xte_full = sc.transform(imp.transform(X[ite]))
            red = RFThreshold(n_trees=100 if FAST else 500, seed=SEED + rep, n_jobs=N_JOBS).fit(Xtr_full, ya)
            Xtr_red, Xte_red = red.transform(Xtr_full), red.transform(Xte_full)
            w = slice_weights(ya)
            if w is not None:
                hi = ya >= 30
                print(f"    weights: min {w.min():.2f} max {w.max():.2f}; rows >= 30 m carry {w[hi].sum()/w.sum():.0%} of the "
                      f"total weight (unweighted {hi.mean():.0%})")
            pd.DataFrame({"rep": rep, "fold": k, "feature": names[red.keep_]}).to_csv(
                f"{OUT}/parts/r{rep}_f{k}_features.csv", index=False)
            print(f"rep {rep} fold {k}: train {len(ya)} (of which {len(ya) - len(itr)} extra) / val {len(ite)} | step 1 kept {len(red.keep_)} of "
                  f"{X.shape[1]} features{' (FALLBACK to top-20)' if red.fallback_ else ''} | prep {time.time()-t0:.0f}s")
            for m in todo:
                make, kind = zoo[m]
                A, B = (Xtr_red, Xte_red) if kind == "red" else (Xtr_full, Xte_full)
                t1 = time.time()
                est = make()
                use_w = w is not None and accepts_weights(est)
                if w is not None and not use_w:
                    print(f"    note: {m} does not accept sample_weight in this scikit-learn version, fitted UNWEIGHTED")
                est = est.fit(A, ya, sample_weight=w) if use_w else est.fit(A, ya)
                pred = np.clip(est.predict(B), 0, None)
                pd.DataFrame({"sample_id": tr.sample_id.to_numpy()[ite], "rep": rep, "fold": k, "model": m, "weighted": bool(use_w),
                              "pred": pred, "fit_s": time.time() - t1, "n_feat": A.shape[1]}
                             ).to_csv(f"{OUT}/parts/r{rep}_f{k}_{m}.csv", index=False)
                print(f"    {m:13s} {time.time()-t1:6.1f}s  ({A.shape[1]} features)")
    parts = [pd.read_csv(f"{OUT}/parts/r{r}_f{k}_{m}.csv") for r in range(N_REPEATS)
             for k in range(N_FOLDS) for m in zoo if os.path.exists(f"{OUT}/parts/r{r}_f{k}_{m}.csv")]
    oof = pd.concat(parts, ignore_index=True)
    oof.to_csv(f"{OUT}/oof_predictions_long.csv", index=False)
    return oof


def summarise(oof, tr, te):
    fr = population_fractions(te, sorted(tr.city.unique()))
    if not fr:
        print("WARNING: no test rows, weighted RMSE is NaN")
    lab = tr[["sample_id", "city", "height_m"]].rename(columns={"height_m": "y"})
    d = oof.merge(lab, on="sample_id", how="left").rename(columns={"pred": "p"})
    pooled, foldrows, binrows = [], [], []
    for (m, rep), g in d.groupby(["model", "rep"]):
        pooled.append(dict(model=m, rep=rep, **metric_row(g.y.to_numpy(), g.p.to_numpy(), g.city.to_numpy(), fr)))
        binrows.append(per_bin_table(g).assign(model=m, rep=rep))
        for k, h in g.groupby("fold"):
            foldrows.append(dict(model=m, rep=rep, fold=k,
                                 **metric_row(h.y.to_numpy(), h.p.to_numpy(), h.city.to_numpy(), fr)))
    pooled, fold_df, bins = pd.DataFrame(pooled), pd.DataFrame(foldrows), pd.concat(binrows)
    keys = ["RMSE", "RMSE_weighted", "RMSE_macro", "MAE"]
    summ = pooled.groupby("model")[keys].mean()
    for kx in ["RMSE", "RMSE_weighted", "RMSE_macro"]:
        summ[kx + "_sd"] = fold_df.groupby("model")[kx].std()      # sd across repeats x folds
    summ["fit_s_mean"] = oof.groupby("model").fit_s.mean()
    summ["n_feat"] = oof.groupby("model").n_feat.mean()
    if "weighted" in oof.columns:
        summ["weighted_fit"] = oof.groupby("model").weighted.mean()      # 1 = fitted with sample weights, 0 = model has no sample_weight
    summ = summ.sort_values("RMSE")
    # per-city weighted RMSE (population frequencies differ by city)
    cityrows = []
    for (m, c), g in d.groupby(["model", "city"]):
        if c in fr:
            fine = np.digitize(g.y.to_numpy(), FINE_EDGES)
            cityrows.append(dict(model=m, city=c, RMSE=float(np.sqrt(np.mean((g.p - g.y) ** 2))),
                                 RMSE_weighted=float(np.sqrt(_wmse_city(g.y.to_numpy(), g.p.to_numpy(), fine, fr[c])))))
    city_df = pd.DataFrame(cityrows).pivot(index="model", columns="city")
    bins_mean = (bins.groupby(["model", "bin", "bin_i"])
                 .agg(n=("n", "mean"), rmse=("rmse", "mean"), bias=("bias", "mean"), mean_ref=("mean_ref", "mean"),
                      mean_pred=("mean_pred", "mean"), mape=("mape", "mean")).reset_index())
    # bias and RMSE per model x city x height class: the diagnostic for labels that were built differently per city
    # (e.g. Belo Horizonte and Niteroi use a higher nDSM statistic than the other cities)
    d["cls"] = np.digitize(d.y.to_numpy(), CLASS_EDGES)
    d["err"] = d.p - d.y
    cc = (d.groupby(["model", "city", "cls"]).err
          .agg(n="size", bias="mean", rmse=lambda e: float(np.sqrt(np.mean(e ** 2)))).reset_index())
    cc["class"] = cc.cls.map(dict(enumerate(CLASS_NAMES)))
    cc.drop(columns="cls").round(3).to_csv(f"{OUT}/per_city_class.csv", index=False)
    d.drop(columns=["cls", "err"]).to_csv(f"{OUT}/oof_with_labels.csv", index=False)      # sample_id, city, y (reference), p (prediction), model, rep, fold
    summ.round(3).to_csv(f"{OUT}/summary_metrics.csv")
    fold_df.to_csv(f"{OUT}/fold_metrics.csv", index=False)
    bins_mean.to_csv(f"{OUT}/per_height_bin.csv", index=False)
    city_df.round(3).to_csv(f"{OUT}/per_city_metrics.csv")
    return summ, fold_df, bins_mean, city_df, fr, d, cc


# ------------------- charts --------------------------------------------
def _style(ax):
    ax.set_facecolor(SURF)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True)


def make_charts(summ, bins_mean, fr, tr):
    plt.rcParams.update({"figure.facecolor": SURF, "axes.labelcolor": INK2, "text.color": INK,
                         "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold"})
    order = list(summ.index)

    # Fig 1: three RMSE views, same model order, error bars = sd across repeats x folds
    fig, axes = plt.subplots(1, 3, figsize=(14, 0.55 * len(order) + 2.2), sharey=True)
    spec = [("RMSE", "RMSE (stratified sample, as in Frantz RMSE)"),
            ("RMSE_weighted", "Frequency-weighted RMSE (Frantz RMSE')"),
            ("RMSE_macro", "Macro RMSE (mean of 5 height classes)")]
    ypos = np.arange(len(order))[::-1]
    for ax, (k, title) in zip(axes, spec):
        _style(ax); ax.grid(axis="y", visible=False)
        vals, sds = summ[k].reindex(order).to_numpy(), summ[k + "_sd"].reindex(order).to_numpy()
        ax.barh(ypos, vals, height=0.62, color=[SLOT[m] for m in order], edgecolor=SURF, linewidth=2)
        ax.errorbar(vals, ypos, xerr=np.nan_to_num(sds), fmt="none", ecolor=INK2, elinewidth=1, capsize=2)
        for yy, v, s in zip(ypos, vals, np.nan_to_num(sds)):
            ax.text(v + s + 0.05 * np.nanmax(vals), yy, f"{v:.2f}", va="center", fontsize=9, color=INK)
        ax.set_title(title, loc="left"); ax.set_xlabel("metres")
        ax.set_xlim(0, np.nanmax(vals + np.nan_to_num(sds)) * 1.2)
    axes[0].set_yticks(ypos); axes[0].set_yticklabels(order)
    fig.suptitle("Spot-check: out-of-fold error, spatial-block CV (mean over repeats; bars = sd across folds)",
                 x=0.01, ha="left", fontsize=11, color=INK2)
    fig.tight_layout(); fig.savefig(f"{OUT}/fig1_rmse_overview.png", dpi=160); plt.show()

    # Fig 2: reference height distributions (top) and RMSE per height bin (bottom)
    fig = plt.figure(figsize=(12, 7.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 3], hspace=0.12)
    a0 = fig.add_subplot(gs[0]); a1 = fig.add_subplot(gs[1], sharex=a0)
    nb = len(FINE_NAMES)
    tr_share = np.bincount(np.digitize(tr.height_m.to_numpy(), FINE_EDGES), minlength=nb) / len(tr)
    nat_share = np.mean([v for v in fr.values()], axis=0) if fr else np.zeros(nb)
    xs = np.arange(nb)
    _style(a0)
    a0.bar(xs - 0.2, tr_share, 0.38, color="#9a9992", edgecolor=SURF, linewidth=2, label="CV sample (stratified)")
    a0.bar(xs + 0.2, nat_share, 0.38, color="#3a3935", edgecolor=SURF, linewidth=2, hatch="///",
           label="natural frequency (test rows)")
    a0.tick_params(labelbottom=False); a0.set_ylabel("share"); a0.legend(frameon=False, fontsize=9)
    a0.set_title("Reference height distribution", loc="left")
    _style(a1)
    ok = bins_mean.groupby("bin_i").n.mean() * N_FOLDS >= MIN_BIN_N
    for m in order:
        g = bins_mean[bins_mean.model == m].sort_values("bin_i")
        g = g[g.bin_i.map(ok).fillna(False)]
        a1.plot(g.bin_i, g.rmse, color=SLOT[m], marker=MARK[m], ls=LINE[m], lw=2, ms=6, label=m)
    a1.set_xticks(xs); a1.set_xticklabels(FINE_NAMES); a1.set_xlabel("reference building height (m)")
    a1.set_ylabel("RMSE (m)"); a1.set_title("RMSE per reference height bin", loc="left")
    a1.legend(frameon=False, fontsize=9, ncol=2, loc="upper left")
    fig.savefig(f"{OUT}/fig2_rmse_by_height.png", dpi=160, bbox_inches="tight"); plt.show()

    # Fig 3: mean prediction vs mean reference per bin (saturation / regression to the mean)
    fig, ax = plt.subplots(figsize=(6.6, 6)); _style(ax)
    top = 0
    for m in order:
        g = bins_mean[bins_mean.model == m].sort_values("bin_i")
        g = g[g.bin_i.map(ok).fillna(False)]
        ax.plot(g.mean_ref, g.mean_pred, color=SLOT[m], marker=MARK[m], ls=LINE[m], lw=2, ms=6, label=m)
        top = max(top, g.mean_ref.max())
    ax.plot([0, top], [0, top], color=INK2, lw=1, ls=(0, (6, 3)), label="1:1")
    ax.set_xlabel("mean reference height in bin (m)"); ax.set_ylabel("mean predicted height (m)")
    ax.set_title("Saturation: mean prediction per reference bin", loc="left")
    ax.legend(frameon=False, fontsize=9, ncol=2)
    fig.savefig(f"{OUT}/fig3_saturation.png", dpi=160, bbox_inches="tight"); plt.show()


def run_spot_check():
    os.makedirs(OUT, exist_ok=True)
    tr, te, feats, ex = load_spot_table()
    oof = run_cv(tr, feats, ex)
    summ, fold_df, bins_mean, city_df, fr, d, cc = summarise(oof, tr, te)
    sel = pd.concat([pd.read_csv(f"{OUT}/parts/r{r}_f{k}_features.csv")
                     for r in range(N_REPEATS) for k in range(N_FOLDS)
                     if os.path.exists(f"{OUT}/parts/r{r}_f{k}_features.csv")])
    freq = sel.groupby("feature").size().sort_values(ascending=False) / sel[["rep", "fold"]].drop_duplicates().shape[0]
    freq.rename("selection_frequency").to_csv(f"{OUT}/step1_selection_frequency.csv")
    show = summ[[c for c in ["RMSE", "RMSE_sd", "RMSE_weighted", "RMSE_weighted_sd", "RMSE_macro", "MAE", "n_feat", "fit_s_mean", "weighted_fit"] if c in summ.columns]].round(2)
    print("\nOut-of-fold metrics, mean over repeats (sd = across repeats x folds); lower is better")
    print(show.to_string())
    print(f"\nStep 1 kept {sel.groupby(['rep','fold']).size().mean():.0f} features per fold on average; "
          f"{int((freq == 1).sum())} features were kept in every fold.")
    best = summ.RMSE_macro.idxmin()
    for what in ("bias", "rmse"):
        print(f"\n{what.upper()} per city and height class, {best} (best macro RMSE), metres" + ("; bias = prediction minus reference" if what == "bias" else ""))
        print(cc[cc.model == best].pivot(index="city", columns="class", values=what)[[c for c in CLASS_NAMES if c in set(cc["class"])]].round(2).to_string())
    make_charts(summ, bins_mean, fr, tr)
    return dict(per_city_class=cc, summary=summ, folds=fold_df, bins=bins_mean, per_city=city_df, oof=d, selection_freq=freq)


RUN = True
if RUN:
    spot = run_spot_check()